# mcp

> Route stdio MCP requests to rustygate gateways

In [ ]:
#| default_exp mcp

Claude Code starts `clikernel-mcp` for each conversation. Its `Router` accepts MCP messages over stdio and sends tool calls to rustygate gateways. Rustygate implements the tools. The router gets their schemas from the local gateway and adds `host` to `list_kernels`, `use_kernel`, and `create`.

Each host has a separate gateway session. The router remembers one current host for calls without an explicit `host`. It removes `host` before forwarding a call but preserves the JSON-RPC request id. Cancellation notifications also retain their original ids.

`main` runs the router and closes its gateway sessions on exit. Closing a session sends an HTTP DELETE. The gateway then stops the kernels that session created with autoclose. If the router started a child gateway, it stops that process too.


In [ ]:
#| export
import asyncio
from fastcore.utils import *
from fastcore.script import call_parse, store_true
from mcpmini.core import serve_stdio, jresp, jerr
from clikernel.core import Gateway, default_gateway, resolve, session_defaults
from clikernel import __version__


In [ ]:
from fastcore.test import *
import os, re, tempfile
from mcpmini.core import jreq, jtool
from rustygate.tools import start_gateway

In [ ]:
#| export
HOST_PARAM = {'type': 'string', 'description': 'Gateway to target: a gateways.toml name, or empty for the default local gateway'}
HOSTED = ('list_kernels', 'use_kernel', 'create')

class Router:
    "Route stdio MCP to rustygate with one session per host and one current host."
    def __init__(self,
        cfgdir=None,  # Config dir for `session_defaults` and `gateways.toml` (the standard one if None)
        quiet=False,  # Keep startup output out of replies?
    ):
        self.cfgdir,self.quiet,self.sessions,self.cur,self.child = cfgdir,quiet,{},'',None

    async def session(self, host=''):
        "Return the session for `host`, initializing it on first use. Empty `host` selects the default local gateway."
        if host not in self.sessions:
            if host:
                url, token, verify = resolve(host, self.cfgdir)
                self.sessions[host] = await Gateway(url, token, verify).initialize(session_defaults(self.cfgdir, self.quiet, local=False))
            else: self.sessions[host], self.child = await default_gateway(self.cfgdir, self.quiet)
        return self.sessions[host]

    async def aclose(self):
        "Close all sessions and their autoclose kernels, then stop any owned gateway."
        try: results = await asyncio.gather(*(s.aclose() for s in self.sessions.values()), return_exceptions=True)
        finally:
            if self.child: self.child.stop()
        errors = [r for r in results if isinstance(r, BaseException)]
        if errors: raise BaseExceptionGroup('MCP session cleanup failed', errors)


`dispatch` answers `initialize` and `ping` itself. The initialization response uses the client's protocol version and the local gateway's instructions. The router maintains its own session with that gateway, separate from the stdio client's session.

`tools/list` returns the local gateway's tools with the added `host` parameters. `tools/call` removes `host` and forwards the remaining message to the chosen gateway. A `use_kernel` or `create` call with an explicit `host` also changes the current host. `list_kernels` doesn't change it.

Cancellation notifications go to the current gateway. Preserving the original request id lets that gateway identify the request. The router does not track the gateway for each outstanding request. Switching hosts before cancellation can therefore send the notification to a different gateway.

In [ ]:
#| export
@patch
async def tools(self:Router):
    "Return the local gateway's tools with `host` added to `list_kernels`, `use_kernel`, and `create`."
    ts = await (await self.session()).tools()
    for t in ts:
        if t['name'] in HOSTED: t['inputSchema'].setdefault('properties', {})['host'] = dict(HOST_PARAM)
    return ts

@patch
async def dispatch(self:Router, msg, requester=None):
    "Answer initialization and ping locally. Forward tool calls to gateways."
    method,id = msg.get('method'), msg.get('id')
    try:
        if method == 'initialize':
            info = (await self.session()).info
            return jresp(id, dict(protocolVersion=msg['params'].get('protocolVersion', '2025-06-18'), capabilities=dict(tools={}),
                serverInfo=dict(name='clikernel', version=__version__), instructions=info.get('instructions')))
        if id is None:
            if method == 'notifications/cancelled': await (await self.session(self.cur)).tr.send(msg)
            return None
        if method == 'ping': return jresp(id, {})
        if method == 'tools/list': return jresp(id, dict(tools=await self.tools()))
        if method == 'tools/call':
            args = msg['params'].setdefault('arguments', {})
            has_host = 'host' in args
            host = args.pop('host', '') or ''
            s = await self.session(host if has_host else self.cur)
            if has_host and msg['params']['name'] in ('use_kernel', 'create'): self.cur = host
            return await s.tr.send(msg)
        return jerr(id, -32601, f'method not found: {method}')
    except Exception as e: return None if id is None else jerr(id, -32603, str(e))

These examples start a disposable rustygate process on a free local port. A temporary `startup.py` defines `base = 42` and prints `ready`. The router's initialization response includes rustygate's instructions:


In [ ]:
g = start_gateway()
os.environ['CLIKERNEL_HOST'] = g.url
cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('base = 42; print("ready")')
router = Router(cfgd)
init = await router.dispatch(jreq('initialize', 1, protocolVersion='2025-11-25', capabilities={}, clientInfo=dict(name='demo', version='0')))
test_eq(init['result']['serverInfo']['name'], 'clikernel')
init['result']['instructions']


"py runs Python/IPython; lua runs bundled Luau. Either starts its language's kernel when none is current, stopped at session end. There is one current kernel; a language mismatch errors without switching. create accepts language=python or luau; an omitted language reuses an existing binding or defaults to Python for a new one. A dlgname execution override applies to one call only. Python startup/inspectors never run in Luau. For native API usage and examples, run lua with code=help(); help('ex.edit_file') returns function details."

In [ ]:
def txt(r): return ''.join(c.get('text','') for c in r['result']['content'] if c['type'] == 'text')
listed = (await router.dispatch(jreq('tools/list', 2)))['result']['tools']
test_eq([t['name'] for t in listed], ['list_kernels', 'create', 'use_kernel', 'delete_kernel', 'py', 'lua', 'restart', 'interrupt'])
byname = {t['name']: t for t in listed}
assert all('host' in byname[n]['inputSchema']['properties'] for n in HOSTED)
assert 'host' not in byname['py']['inputSchema']['properties']
r = await router.dispatch(jtool('py', 3, code='base'))
assert 'created kernel' in txt(r) and 'ready' in txt(r) and txt(r).endswith('42')
txt(r)


'created kernel 95c8606cb5e24e9a80096bb3c0d4a593 language=python\nready\n42'

`gateways.toml` maps host names to gateways. Here, `alt` refers to a second disposable gateway. Creating a kernel there makes it the router's current host. The following Python calls omit `host` and run in that kernel:

In [ ]:
g2 = start_gateway()
(cfgd/'gateways.toml').write_text(f'[gateways.alt]\nurl = "{g2.url}"\n')
r = await router.dispatch(jtool('create', 4, dlgname='far.ipynb', host='alt'))
assert 'created kernel' in txt(r) and 'ready' in txt(r)
await router.dispatch(jtool('py', 5, code='marker = 7'))
test_eq(txt(await router.dispatch(jtool('py', 6, code='marker'))), '7')

Use `host=''` to address the default local gateway. Listing its kernels doesn't change the current host. Selecting its kernel with `use_kernel` does.

The local kernel has `base` from startup but no `marker`. The `alt` gateway still has the kernel bound to `far.ipynb`:

In [ ]:
kid = re.search(r'^(\w+) ', txt(await router.dispatch(jtool('list_kernels', 7, host=''))), re.M).group(1)
await router.dispatch(jtool('use_kernel', 8, kernel=kid, host=''))
assert 'NameError' in txt(await router.dispatch(jtool('py', 9, code='marker')))
test_eq(txt(await router.dispatch(jtool('py', 10, code='base'))), '42')
txt(await router.dispatch(jtool('list_kernels', 11, host='alt')))

'834ed54e99744a0c96648144f3dc56fd  alive  language=python  connections=1  dlgname=far.ipynb  <- current'

Rustygate returns this PIL image as a text block followed by an MCP image block. It follows `fastcore.nbio.IMG_MIMES` when choosing a format. PIL supplies PNG and JPEG representations, and that preference order selects JPEG. The router passes both content blocks to the client unchanged:


In [ ]:
r = await router.dispatch(jtool('py', 12, code='from PIL import Image\nImage.new("RGB", (300, 200), "red")'))
blocks = r['result']['content']
test_eq([b['type'] for b in blocks], ['text', 'image'])
test_eq(blocks[1]['mimeType'], 'image/jpeg')
{k: (v[:20] + '…' if k == 'data' else v) for k,v in blocks[1].items()}


{'type': 'image', 'data': '/9j/4AAQSkZJRgABAQAA…', 'mimeType': 'image/jpeg'}

`create(language='luau')` selects a Luau kernel on the chosen gateway. Luau doesn't run Python startup code. A `py` call against that kernel returns an error without changing the kernel selection or its state:


In [ ]:
r = await router.dispatch(jtool('create', 20, dlgname='native.ipynb', language='luau', host='alt'))
assert not r['result']['isError'] and 'language=luau' in txt(r)
test_eq(txt(await router.dispatch(jtool('lua', 21, code='saved=42; return saved, base == nil'))), '42\ttrue')
wrong = await router.dispatch(jtool('py', 22, code='saved=0'))
assert wrong['result']['isError'] and 'not python' in txt(wrong)
test_eq(txt(await router.dispatch(jtool('lua', 23, code='saved'))), '42')
txt(wrong)

'kernel 64632da2f829435f93bd87c8bbf0ea5e uses luau, not python; select a matching kernel with use_kernel or create'

`restart` clears the Luau kernel's state, including `saved`. It still doesn't run Python startup, which would define `base`. Switching back to the original Python kernel reveals its unchanged state:

In [ ]:
await router.dispatch(jtool('restart', 24))
test_eq(txt(await router.dispatch(jtool('lua', 25, code='return saved, base'))), 'nil\tnil')
await router.dispatch(jtool('use_kernel', 26, kernel=kid, host=''))
test_eq(txt(await router.dispatch(jtool('py', 27, code='base'))), '42')
txt(await router.dispatch(jtool('list_kernels', 28, host='alt')))

'64632da2f829435f93bd87c8bbf0ea5e  alive  language=luau  connections=1  dlgname=native.ipynb  <- current\n834ed54e99744a0c96648144f3dc56fd  alive  language=python  connections=0  dlgname=far.ipynb'

`main` serves one router over stdio. The router opens each gateway session on first use. Initialization opens the local session, starting a child gateway if necessary. A later `tools/list` request fetches the tool schemas.

On exit, `main` closes all gateway sessions and stops any child gateway it owns. Stopping that gateway also stops its kernels. `--quiet` suppresses kernel startup output in replies.


In [ ]:
#| export
@call_parse
def main(
    quiet:store_true=False,  # Keep startup output out of replies
):
    "Run `clikernel-mcp` as a stdio server."
    async def _main():
        router = Router(quiet=quiet)
        try: await serve_stdio(router)
        finally: await router.aclose()
    asyncio.run(_main())


We can also use the installed `clikernel-mcp` command through an MCP stdio client. It exposes `py` and `lua`, but no `bash` tool. This example runs the same calculation in Python and Luau:

In [ ]:
import shutil
from mcpmini.core import MCPClient


In [ ]:
cmd = shutil.which('clikernel-mcp')
assert cmd, 'clikernel-mcp script not installed: run `uv sync`'
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(Path(tempfile.mkdtemp()))}
async with MCPClient.stdio([cmd], env=env) as m:
    assert 'py' in dir(m.tools) and 'lua' in dir(m.tools) and 'bash' not in dir(m.tools)
    res = await m.tools.py(code='6*7')
    await m.tools.create(dlgname='stdio-native.ipynb', language='luau')
    native_res = await m.tools.lua(code='6*7')
assert res.endswith('42') and native_res == '42'
native_res


'42'

`router.aclose()` ends its sessions on both gateways. Each gateway stops the kernels this router created with autoclose. Selecting an existing kernel does not make the router responsible for stopping it.

If a session close raises, the router still attempts the other sessions and stops its own child gateway. It raises the session errors together after cleanup.

The gateways in these tests were already running when the router connected. The router owns neither process. After closing its sessions, we use a new router to check that both gateways have no remaining kernels:

In [ ]:
await router.aclose()
assert router.child is None
chk = Router(cfgd)
near,far = [txt(await chk.dispatch(jtool('list_kernels', i, host=h))) for i,h in ((1, ''), (2, 'alt'))]
await chk.aclose()
test_eq((near, far), ('no kernels', 'no kernels'))
near,far


('no kernels', 'no kernels')

## A live client

These optional tests use Claude Code's MCP client through the headless Agent SDK. They spend model tokens and have `#| eval: false` to exclude them from automated runs. Run them manually after changing the tools or updating Claude Code.

The first test asks Claude to call `py` on the live gateway. That call starts a kernel automatically:


In [ ]:
#| eval: false
import logging, random
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage

In [ ]:
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_dir = Path(tempfile.mkdtemp())
lenv = {k:str(v) for k,v in (os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(live_dir)}).items()}
opts = ClaudeAgentOptions(mcp_servers=dict(ck=dict(type='stdio', command=cmd, env=lenv)), cwd=str(live_dir),
    allowed_tools=['mcp__ck__py'], max_turns=6)
prompt = 'Using the ck MCP py tool, compute 17*19 in the kernel. Reply with just the number.'
msgs = [m async for m in query(prompt=prompt, options=opts)]
tus = {b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if isinstance(b, ToolUseBlock)}
res = first(m.result for m in msgs if isinstance(m, ResultMessage))
assert 'mcp__ck__py' in tus
assert '323' in res
res


'323'

The next test checks image delivery through the router, stdio transport, and Claude Code's client. It chooses a random color without naming it in the prompt. The requested Python code reads the color from a file and displays a plain image. Claude must identify the color from that image. The final assertion compares its answer with the chosen color.

In [ ]:
#| eval: false
color = random.choice(['red', 'green', 'blue', 'yellow', 'purple', 'orange'])
(live_dir/'color.txt').write_text(color)
code = "from PIL import Image\nImage.new('RGB', (200,200), open('color.txt').read().strip())"
prompt = ('Using the ck MCP tools: run exactly this code with the py tool (do not run anything else, and do not read color.txt any other way):\n\n'
    f'{code}\n\nThe py result includes an image. Reply with just the color of that image.')
msgs2 = [m async for m in query(prompt=prompt, options=opts)]
res2 = first(m.result for m in msgs2 if isinstance(m, ResultMessage))
assert color in res2.lower()
color, res2

('purple', 'Purple.')

In [ ]:
#|hide
g.stop()
g2.stop()
del os.environ['CLIKERNEL_HOST']


In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()